# Current Kaggle runtime setup

This setup-only notebook inventories the actual Linux kernel and prepares one isolated CPU, DINO, LocateAnything/SAM or publication environment. It never opens a scene, anomaly label or scientific evaluator and never calls a model. It preserves exact package pins while recording the observed Python 3.12 patch before the new method freeze.

The source remains a verified working-directory snapshot. Use a separate job for each profile. CPU setup does not need a GPU; CUDA wheels can also be prepared without allocating one. Region inference later requires the configured T4 accelerator. Retain the wheelhouse and runtime-profile output before starting downstream stages.

In [ ]:
from pathlib import Path
import json, os
try:
    NOTEBOOK_RELATIVE = 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb'
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)

    SOURCE = _runtime.verify_source(CODE_ROOT, NOTEBOOK_RELATIVE)
    sys.path.insert(0, str(CODE_ROOT / 'scripts'))
    print({'source_snapshot_sha256': SOURCE['snapshot_sha256'], 'scientific_imports_started': False})
except BaseException as error:
    receipt_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    receipt_root.mkdir(parents=True, exist_ok=True)
    temporary = receipt_root/'setup-bootstrap.json.partial'
    temporary.write_text(json.dumps({'status':'setup_bootstrap_failed','error_type':type(error).__name__,'scientific_run_started':False})+'\n')
    os.replace(temporary, receipt_root/'setup-bootstrap.json')
    raise

## Reviewed setup configuration

Set `VPI_NEXT_CONFIG` to a setup JSON containing `profile` (`cpu`, `dino`, `dino-gpu`, `regions` or `tracking`), `action` (`probe` or `prepare`), `output_root`, `target`, and `wheelhouse`. Set `build_wheelhouse=true` to build the exact offline artifacts on Linux from the source lock. The default action only records metadata. Retained outputs belong under `/kaggle/working`; dependency-only targets use fresh `/tmp/vpi-runtime-*` directories so thousands of installed files are not retained as notebook output. GPU and DINO use separate targets because their Torch/Transformers versions differ.

In [ ]:
CONFIG = _runtime.resolve_config('runtime-setup', CODE_ROOT)
PROFILE_NAMES = {'cpu': 'linux-cpu-cache-cp312', 'dino': 'linux-dino-fp32-cpu', 'dino-gpu': 'linux-dino-fp32-cuda126', 'regions': 'linux-regions-cuda126', 'tracking': 'linux-tracking-cp312'}
if CONFIG.get('profile') not in PROFILE_NAMES or CONFIG.get('action', 'probe') not in {'probe', 'prepare'}:
    raise ValueError('Choose one registered profile and probe/prepare action')
profile_path = CODE_ROOT / 'requirements/profiles' / (PROFILE_NAMES[CONFIG['profile']] + '.json')
output_root = _runtime.writable_output(CONFIG['output_root'])
from kaggle_setup_probe import probe
PROBE = probe([profile_path])
_runtime.atomic_json(output_root / 'initial_setup_probe.json', PROBE)
print(PROBE)

## Build verified wheels and private runtime

Wheel builds are bounded and use official PyPI/PyTorch indexes. Only dependencies are resolved or built; nothing is installed into the notebook kernel. The next step installs the verified wheelhouse into a fresh private target and seals its exact package metadata and observed Python patch. This is setup verification, not scientific or model compatibility evidence.

In [ ]:
if CONFIG.get('action', 'probe') == 'prepare':
    wheelhouse = Path(CONFIG['wheelhouse'])
    if CONFIG.get('build_wheelhouse', False):
        wheelhouse = _runtime.writable_output(wheelhouse)
        if CONFIG['profile'] == 'cpu':
            from prepare_linux_wheelhouse import prepare as build_wheels
        else:
            from prepare_neural_wheelhouse import prepare as build_wheels
        print(build_wheels(profile_path, wheelhouse))
    from prepare_runtime_environment import prepare_isolated
    base_profile = CODE_ROOT / 'requirements/current-evaluation-runtime.json' if CONFIG['profile'] == 'cpu' else profile_path
    RECEIPT = prepare_isolated(base_profile, wheelhouse, _runtime.runtime_target(CONFIG['target']), output_root / 'runtime-profile')
    print(RECEIPT)
else:
    print({'status': 'metadata_only', 'dependencies_installed': False})

## Optional pinned model assets

Set `download_models=true` only in a DINO or regions setup job, plus `model_asset_root` and `models_only=all|regions|dino`. The bounded model downloader uses the existing immutable model revisions and checkpoint hashes. The outer timeout covers each selected model's 1,800-second download budget plus checksum time; timeout cleanup stops the whole child process group. It downloads assets without constructing or calling a model. Reuse its output in inference jobs.

In [ ]:
if CONFIG.get('download_models', False):
    if CONFIG.get('action') != 'prepare' or CONFIG['profile'] not in {'dino', 'dino-gpu', 'regions'}:
        raise ValueError('Model asset download requires a prepared DINO or regions setup profile')
    model_root = _runtime.writable_output(CONFIG['model_asset_root'])
    print(_runtime.run_model_asset_preparation(CODE_ROOT, model_root, CONFIG.get('models_only', 'dino' if CONFIG['profile'] == 'dino-gpu' else CONFIG['profile']), CONFIG['target'], per_model_timeout=1800))
else:
    print({'model_asset_download': 'not requested'})

## Downstream binding

A downstream job sets `VPI_RUNTIME_PROFILE_ROOT` to the sealed `runtime-profile` resource, `VPI_WHEELHOUSE` to the matching wheelhouse, `VPI_PACKAGE_TARGET` to a fresh target and `VPI_INSTALL_DEPENDENCIES=1` before its normal bootstrap. DINO CPU/GPU jobs set their matching exact `VPI_BASE_RUNTIME_PROFILE`; the region notebook selects its GPU profile explicitly.

The bootstrap verifies unchanged package/device pins, the base-profile hash and the observed Linux Python patch. The subsequent method freeze records the actual scientific runtime. A later runtime change cannot bypass a completed freeze. Run development compatibility checks before protected evaluation. Pinned model assets may be prepared by the optional bounded download cell above; no model inference is run.